In [17]:
!pip install -q ultralytics
import ultralytics

In [18]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
from ultralytics import YOLO

In [20]:
!pip install -q kaggle
from google.colab import userdata

kaggle_token = userdata.get("KAGGLE_API_TOKEN")
import os

os.environ["KAGGLE_API_TOKEN"] = kaggle_token
!kaggle datasets list -s bdd100k | head
!mkdir -p /content/bdd100k

!kaggle datasets download \
    -d alvaromalfaro/bdd100k \
    -p /content/bdd100k

import os
from pathlib import Path

LABEL_ROOT = "/content/bdd100k/extracted/bdd100k/labels"

ref                                                       title                                                  size  lastUpdated                 downloadCount  voteCount  usabilityRating  
--------------------------------------------------------  ----------------------------------------------  -----------  --------------------------  -------------  ---------  ---------------  
solesensei/solesensei_bdd100k                             bdd100k                                          8166127525  2019-05-21 21:10:21.507000          45536        157  0.75             
marquis03/bdd100k                                         BDD100K Images                                   5781166420  2023-12-15 14:12:18.997000           3671         20  1                
marquis03/bdd100k-weather-classification                  BDD100K Weather Classification                   5662942691  2023-12-15 14:28:34.747000           1239         24  1                
a7madmostafa/bdd100k-yolo                    

In [21]:
from pathlib import Path
import json
import os
import random
import shutil
from collections import Counter, defaultdict

BDD_ROOT = Path("/content/bdd100k/extracted/bdd100k")

TRAIN_LABELS = BDD_ROOT / "labels/train"
VAL_LABELS   = BDD_ROOT / "labels/val"

TRAIN_IMAGES = BDD_ROOT / "images/100k/train"
VAL_IMAGES   = BDD_ROOT / "images/100k/val"

OUTPUT_ROOT = Path("/content/BDD5")

CLASSES = {
    "car": 0,
    "bus": 1,
    "person": 2,
    "truck": 3,
    "bike": 4
}

CLASS_NAMES = list(CLASSES.keys())

print("Train labels:", len(list(TRAIN_LABELS.glob("*.json"))))
print("Val labels:", len(list(VAL_LABELS.glob("*.json"))))
print("Classes:", CLASSES)

Train labels: 0
Val labels: 0
Classes: {'car': 0, 'bus': 1, 'person': 2, 'truck': 3, 'bike': 4}


In [22]:
# ============================================================
# BUILD IMAGE -> CLASSES MAP + SELECT SUBSET
# ============================================================

from collections import defaultdict
import json
import random

# ------------------------------------------------------------
# 1. Build image -> set of classes for each split
# ------------------------------------------------------------

def build_image_class_map(label_dir):
    """
    Reads BDD100K JSON label files and creates:

        {
            image_id: {"car", "person"},
            image_id: {"truck"},
            ...
        }

    Only the 5 classes in CLASSES are included.
    """

    image_class_map = {}

    json_files = list(label_dir.glob("*.json"))

    print(f"Reading {len(json_files)} label files from {label_dir}")

    for json_file in json_files:

        image_id = json_file.stem

        try:
            with open(json_file, "r") as f:
                data = json.load(f)

            classes_present = set()

            # BDD100K format used in your notebook
            objects = data["frames"][0]["objects"]

            for obj in objects:

                category = obj.get("category")

                # Keep only our 5 classes
                if category in CLASSES:

                    # Make sure it has a bounding box
                    if obj.get("box2d") is not None:
                        classes_present.add(category)

            # Only keep images containing at least one
            # of our 5 target classes
            if classes_present:
                image_class_map[image_id] = classes_present

        except Exception as e:
            print(f"Error reading {json_file}: {e}")

    return image_class_map


# ------------------------------------------------------------
# 2. Create the missing variables
# ------------------------------------------------------------

train_image_classes = build_image_class_map(TRAIN_LABELS)
val_image_classes = build_image_class_map(VAL_LABELS)


print("\n======================================")
print("IMAGE CLASS MAP CREATED")
print("======================================")

print("Train images containing target classes:",
      len(train_image_classes))

print("Val images containing target classes:",
      len(val_image_classes))


# ------------------------------------------------------------
# 3. Show class distribution
# ------------------------------------------------------------

def print_class_distribution(image_class_map, name):

    counts = Counter()

    for classes_present in image_class_map.values():

        for cls in classes_present:
            counts[cls] += 1

    print(f"\n{name} class distribution:")

    for cls in CLASS_NAMES:
        print(f"  {cls:10s}: {counts[cls]}")


print_class_distribution(
    train_image_classes,
    "TRAIN"
)

print_class_distribution(
    val_image_classes,
    "VAL"
)


# ------------------------------------------------------------
# 4. Image selection function
# ------------------------------------------------------------

def select_images(image_class_map, target_images, seed=42):

    random.seed(seed)

    all_images = list(image_class_map.keys())
    random.shuffle(all_images)

    # Don't request more images than available
    target_images = min(target_images, len(all_images))

    selected = set()

    # --------------------------------------------------------
    # Build class -> images mapping
    # --------------------------------------------------------

    class_to_images = defaultdict(list)

    for image_id, classes_present in image_class_map.items():

        for cls in classes_present:
            class_to_images[cls].append(image_id)

    # --------------------------------------------------------
    # Process rare classes first
    # --------------------------------------------------------

    class_order = sorted(
        CLASS_NAMES,
        key=lambda c: len(class_to_images[c])
    )

    print("\nClass selection order:")

    for cls in class_order:
        print(
            f"  {cls}: "
            f"{len(class_to_images[cls])} images"
        )

    # --------------------------------------------------------
    # Try to give every class representation
    # --------------------------------------------------------

    target_per_class = target_images // len(CLASS_NAMES)

    for cls in class_order:

        candidates = class_to_images[cls].copy()
        random.shuffle(candidates)

        for image_id in candidates:

            if len(selected) >= target_images:
                break

            selected.add(image_id)

            # Count selected images containing this class
            class_selected_count = sum(
                cls in image_class_map[x]
                for x in selected
            )

            if class_selected_count >= target_per_class:
                break

    # --------------------------------------------------------
    # Fill remaining images randomly
    # --------------------------------------------------------

    remaining = [
        x for x in all_images
        if x not in selected
    ]

    random.shuffle(remaining)

    needed = target_images - len(selected)

    selected.update(remaining[:needed])

    return list(selected)[:target_images]


# ------------------------------------------------------------
# 5. Select train and validation images
# ------------------------------------------------------------

train_selected = select_images(
    train_image_classes,
    target_images=2000,
    seed=42
)

val_selected = select_images(
    val_image_classes,
    target_images=500,
    seed=42
)


print("\n======================================")
print("SELECTION COMPLETE")
print("======================================")

print("Selected train images:", len(train_selected))
print("Selected val images:", len(val_selected))

Reading 0 label files from /content/bdd100k/extracted/bdd100k/labels/train
Reading 0 label files from /content/bdd100k/extracted/bdd100k/labels/val

IMAGE CLASS MAP CREATED
Train images containing target classes: 0
Val images containing target classes: 0

TRAIN class distribution:
  car       : 0
  bus       : 0
  person    : 0
  truck     : 0
  bike      : 0

VAL class distribution:
  car       : 0
  bus       : 0
  person    : 0
  truck     : 0
  bike      : 0

Class selection order:
  car: 0 images
  bus: 0 images
  person: 0 images
  truck: 0 images
  bike: 0 images

Class selection order:
  car: 0 images
  bus: 0 images
  person: 0 images
  truck: 0 images
  bike: 0 images

SELECTION COMPLETE
Selected train images: 0
Selected val images: 0


In [23]:
for split in ["train", "val"]:
    (OUTPUT_ROOT / f"images/{split}").mkdir(
        parents=True,
        exist_ok=True
    )

    (OUTPUT_ROOT / f"labels/{split}").mkdir(
        parents=True,
        exist_ok=True
    )

IMAGE_WIDTH = 1280
IMAGE_HEIGHT = 720


def convert_to_yolo(
    image_ids,
    label_dir,
    image_dir,
    output_image_dir,
    output_label_dir
):

    for image_id in image_ids:

        json_file = label_dir / f"{image_id}.json"
        image_file = image_dir / f"{image_id}.jpg"

        if not json_file.exists() or not image_file.exists():
            continue

        with open(json_file, "r") as f:
            data = json.load(f)

        objects = data["frames"][0]["objects"]

        label_lines = []

        for obj in objects:

            category = obj["category"]

            if category not in CLASSES:
                continue

            box = obj.get("box2d")

            if box is None:
                continue

            x1 = box["x1"]
            y1 = box["y1"]
            x2 = box["x2"]
            y2 = box["y2"]

            # Clamp coordinates
            x1 = max(0, min(x1, IMAGE_WIDTH))
            x2 = max(0, min(x2, IMAGE_WIDTH))
            y1 = max(0, min(y1, IMAGE_HEIGHT))
            y2 = max(0, min(y2, IMAGE_HEIGHT))

            # Convert to YOLO format
            xc = ((x1 + x2) / 2) / IMAGE_WIDTH
            yc = ((y1 + y2) / 2) / IMAGE_HEIGHT

            w = (x2 - x1) / IMAGE_WIDTH
            h = (y2 - y1) / IMAGE_HEIGHT

            class_id = CLASSES[category]

            label_lines.append(
                f"{class_id} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}"
            )

        # Only keep images containing at least one target class
        if not label_lines:
            continue

        shutil.copy2(
            image_file,
            output_image_dir / image_file.name
        )

        with open(
            output_label_dir / f"{image_id}.txt",
            "w"
        ) as f:
            f.write("\n".join(label_lines))

In [24]:
convert_to_yolo(
    train_selected,
    TRAIN_LABELS,
    TRAIN_IMAGES,
    OUTPUT_ROOT / "images/train",
    OUTPUT_ROOT / "labels/train"
)

convert_to_yolo(
    val_selected,
    VAL_LABELS,
    VAL_IMAGES,
    OUTPUT_ROOT / "images/val",
    OUTPUT_ROOT / "labels/val"
)
print(
    "Train images:",
    len(list((OUTPUT_ROOT / "images/train").glob("*.jpg")))
)

print(
    "Train labels:",
    len(list((OUTPUT_ROOT / "labels/train").glob("*.txt")))
)

print(
    "Val images:",
    len(list((OUTPUT_ROOT / "images/val").glob("*.jpg")))
)

print(
    "Val labels:",
    len(list((OUTPUT_ROOT / "labels/val").glob("*.txt")))
)

Train images: 0
Train labels: 0
Val images: 0
Val labels: 0


In [25]:
yaml_text = f"""
path: {OUTPUT_ROOT}

train: images/train
val: images/val

names:
  0: car
  1: bus
  2: person
  3: truck
  4: bike
"""

with open(OUTPUT_ROOT / "data.yaml", "w") as f:
    f.write(yaml_text)

print((OUTPUT_ROOT / "data.yaml").read_text())


path: /content/BDD5

train: images/train
val: images/val

names:
  0: car
  1: bus
  2: person
  3: truck
  4: bike



In [26]:
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import random

ID_TO_CLASS = {v: k for k, v in CLASSES.items()}

def show_sample(image_path, label_path):
    image = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(image)

    W, H = image.size

    with open(label_path) as f:
        for line in f:
            cls, xc, yc, w, h = map(float, line.split())

            xc *= W
            yc *= H
            w *= W
            h *= H

            x1 = xc - w / 2
            y1 = yc - h / 2
            x2 = xc + w / 2
            y2 = yc + h / 2

            draw.rectangle([x1, y1, x2, y2], outline="red", width=2)
            draw.text(
                (x1, max(0, y1 - 15)),
                ID_TO_CLASS[int(cls)],
                fill="red"
            )

    plt.figure(figsize=(14, 8))
    plt.imshow(image)
    plt.axis("off")
    plt.show()
train_images = list(
    (OUTPUT_ROOT / "images/train").glob("*.jpg")
)

# for image_path in random.sample(train_images, 6):
#     label_path = OUTPUT_ROOT / "labels/train" / f"{image_path.stem}.txt"
#     show_sample(image_path, label_path)

In [27]:
# from ultralytics import YOLO
# from google.colab import drive
# drive.mount('/content/drive')
# model = YOLO("yolo11s.pt")

# results = model.train(
#     data="/content/BDD5/data.yaml",
#     epochs=50,
#     imgsz=640,
#     batch=16,
#     #device=0,
#     workers=2,
#     project="/content/drive/MyDrive/BDD_runs",
#     name="yolo11s_baseline",
#     pretrained=True,
#     patience=10,
#     seed=42
# )

In [28]:
from pathlib import Path
import json

# ============================================================
# PATHS
# ============================================================

# Change this if your BDD100K validation images are elsewhere
VAL_IMAGES_DIR = Path(
    "/content/drive/MyDrive/BDD100K/images/100k/val"
)

# Folder containing your original BDD100K annotation JSON
# Change this path to wherever your original labels are.
BDD_LABEL_DIR = Path(
    "/content/drive/MyDrive/BDD100K/labels"
)

# Output file that your existing code expects
VAL_JSON = Path(
    "/content/drive/MyDrive/BDD_runs/bdd100k_labels_images_val.json"
)

VAL_JSON.parent.mkdir(parents=True, exist_ok=True)


# ============================================================
# FIND JSON FILES
# ============================================================

json_files = list(BDD_LABEL_DIR.rglob("*.json"))

print("JSON files found:")

for f in json_files:
    print(" ", f)


# ============================================================
# FIND VALIDATION LABEL JSON
# ============================================================

val_json_candidates = [
    f for f in json_files
    if "val" in f.name.lower()
]

if not val_json_candidates:
    raise FileNotFoundError(
        f"No validation JSON found inside {BDD_LABEL_DIR}"
    )

print("\nValidation candidates:")

for f in val_json_candidates:
    print(" ", f)


# Use the first validation annotation file
SOURCE_VAL_JSON = val_json_candidates[0]

print("\nUsing source:")
print(SOURCE_VAL_JSON)


# ============================================================
# LOAD ORIGINAL BDD100K LABELS
# ============================================================

with open(SOURCE_VAL_JSON, "r") as f:
    bdd_val = json.load(f)

print(
    "\nLoaded",
    len(bdd_val),
    "validation annotations"
)


# ============================================================
# CREATE THE JSON YOUR CODE EXPECTS
# ============================================================

output = []

for item in bdd_val:

    image_name = item.get("name")

    if image_name is None:
        continue

    output.append({
        "name": image_name,
        "attributes": item.get("attributes", {})
    })


# ============================================================
# SAVE
# ============================================================

with open(VAL_JSON, "w") as f:
    json.dump(output, f, indent=2)

print("\nCreated:")
print(VAL_JSON)

print("\nNumber of entries:", len(output))

JSON files found:


FileNotFoundError: No validation JSON found inside /content/drive/MyDrive/BDD100K/labels

In [29]:
from pathlib import Path

# Show your actual BDD5 structure
root = Path("/content/BDD5")

for p in root.rglob("*"):
    if p.is_file():
        print(p)

/content/BDD5/data.yaml


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q ultralytics

import json, os, yaml
from pathlib import Path
from collections import defaultdict
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO

# ---- Config: edit these ----
DATA_YAML  = "/content/BDD5/data.yaml"
MODEL_PATH = "/content/drive/MyDrive/BDD_runs/yolo11s_baseline/weights/best.pt"
VAL_JSON   = "/content/drive/MyDrive/BDD_runs/bdd100k_labels_images_val.json"  # upload this
REPORT_DIR = "/content/drive/MyDrive/BDD_runs/yolo11s_baseline/eval_report"
os.makedirs(REPORT_DIR, exist_ok=True)

model = YOLO(MODEL_PATH)

with open(DATA_YAML) as f:
    data_cfg = yaml.safe_load(f)

base_path = Path(data_cfg.get("path", "."))
val_rel = data_cfg["val"]
val_images_dir = Path(val_rel) if os.path.isabs(val_rel) else (base_path / val_rel).resolve()

if val_images_dir.is_file():  # val already a txt list of image paths
    val_image_paths = [Path(p.strip()).resolve() for p in open(val_images_dir) if p.strip()]
else:
    val_image_paths = sorted(val_images_dir.resolve().glob("*.jpg")) + sorted(val_images_dir.resolve().glob("*.png"))

print(f"Found {len(val_image_paths)} val images at {val_images_dir}")

with open(VAL_JSON) as f:
    bdd_val = json.load(f)
attrs_by_name = {item["name"]: item.get("attributes", {}) for item in bdd_val}

image_attrs = {}
for p in val_image_paths:
    a = attrs_by_name.get(p.name)
    if a:
        image_attrs[str(p)] = a

missing = len(val_image_paths) - len(image_attrs)
if missing:
    print(f"Warning: {missing} val images had no match in {VAL_JSON} (filename mismatch?)")

# ============================================================
# 1) Overall + per-class metrics
# ============================================================
print("\n=== Overall / per-class validation ===")
overall = model.val(data=DATA_YAML, split="val", plots=True, project=REPORT_DIR, name="overall")

class_names = overall.names
per_class_rows = []
for i, idx in enumerate(overall.box.ap_class_index):
    per_class_rows.append({
        "class": class_names[int(idx)],
        "precision": overall.box.p[i],
        "recall": overall.box.r[i],
        "mAP50": overall.box.ap50[i],
        "mAP50-95": overall.box.ap[i],
    })
per_class_df = pd.DataFrame(per_class_rows)
per_class_df.to_csv(os.path.join(REPORT_DIR, "per_class_metrics.csv"), index=False)
print(per_class_df)

# ============================================================
# 2) Run val() on an arbitrary image subset
# ============================================================
def eval_subset(image_paths, tmp_name):
    if not image_paths:
        return None
    list_path = os.path.join(REPORT_DIR, f"_{tmp_name}_images.txt")
    with open(list_path, "w") as f:
        f.write("\n".join(str(p) for p in image_paths))

    tmp_cfg = dict(data_cfg)
    tmp_cfg["val"] = list_path
    tmp_yaml_path = os.path.join(REPORT_DIR, f"_{tmp_name}_data.yaml")
    with open(tmp_yaml_path, "w") as f:
        yaml.safe_dump(tmp_cfg, f)

    return model.val(data=tmp_yaml_path, split="val", plots=False,
                      project=REPORT_DIR, name=f"subset_{tmp_name}", verbose=False)

# ============================================================
# 3) Breakdown by weather / scene / timeofday
# ============================================================
def breakdown_by(attr_key):
    buckets = defaultdict(list)
    for path_str, attrs in image_attrs.items():
        buckets[attrs.get(attr_key, "unknown")].append(path_str)

    rows = []
    for value, paths in buckets.items():
        print(f"--- {attr_key} = {value} ({len(paths)} images) ---")
        metrics = eval_subset(paths, f"{attr_key}_{value}".replace(" ", "_"))
        if metrics is None:
            continue
        rows.append({
            attr_key: value,
            "n_images": len(paths),
            "precision": metrics.box.mp,
            "recall": metrics.box.mr,
            "mAP50": metrics.box.map50,
            "mAP50-95": metrics.box.map,
        })
    return pd.DataFrame(rows)

weather_df   = breakdown_by("weather")
scene_df     = breakdown_by("scene")
timeofday_df = breakdown_by("timeofday")

weather_df.to_csv(os.path.join(REPORT_DIR, "weather_metrics.csv"), index=False)
scene_df.to_csv(os.path.join(REPORT_DIR, "scene_metrics.csv"), index=False)
timeofday_df.to_csv(os.path.join(REPORT_DIR, "timeofday_metrics.csv"), index=False)

print("\nWeather:\n", weather_df)
print("\nScene:\n", scene_df)
print("\nTime of day:\n", timeofday_df)

# ============================================================
# 4) Plots
# ============================================================
def plot_metric_bar(df, key_col, metric_col, title, out_path):
    if df.empty:
        return
    df_sorted = df.sort_values(metric_col, ascending=False)
    plt.figure(figsize=(8, 5))
    plt.bar(df_sorted[key_col].astype(str), df_sorted[metric_col], color="#4C72B0")
    plt.xticks(rotation=45, ha="right")
    plt.ylabel(metric_col)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=120)
    plt.close()

plot_metric_bar(per_class_df, "class", "mAP50", "mAP50 by class", os.path.join(REPORT_DIR, "map50_by_class.png"))
plot_metric_bar(weather_df, "weather", "mAP50", "mAP50 by weather", os.path.join(REPORT_DIR, "map50_by_weather.png"))
plot_metric_bar(scene_df, "scene", "mAP50", "mAP50 by scene", os.path.join(REPORT_DIR, "map50_by_scene.png"))
plot_metric_bar(timeofday_df, "timeofday", "mAP50", "mAP50 by time of day", os.path.join(REPORT_DIR, "map50_by_timeofday.png"))

print(f"\nDone. Report artifacts saved to {REPORT_DIR}")


In [ ]:
!ls -lah /content/runs/yolo11m_baseline
